In [0]:
# ===================================================
# BLOCK 1 — FAILURE PARAMETERS (PYTHON)
# ===================================================

"""
Receive the failed workflow run identifier and record an actionable operational
failure state.
"""

dbutils.widgets.text("job_run_id", "MANUAL", "Lakeflow Job Run ID")
dbutils.widgets.text(
    "failure_message",
    "One or more workflow tasks failed. Review the Lakeflow Job run details.",
    "Failure message",
)

JOB_RUN_ID = dbutils.widgets.get("job_run_id")
FAILURE_MESSAGE = dbutils.widgets.get("failure_message")

escaped_failure_message = FAILURE_MESSAGE.replace("'", "''")

In [0]:
# ===================================================
# BLOCK 2 — RECORD FAILED RUN (PYTHON)
# ===================================================

"""
Close the operational audit record as failed while preserving the original
start timestamp and job identity.
"""

spark.sql(
    f"""
    UPDATE semiconplus_portfolio.operations.workflow_run_log
    SET
        run_status = 'FAILED',
        completed_at_utc = CURRENT_TIMESTAMP(),
        failure_message = '{escaped_failure_message}',
        last_updated_at_utc = CURRENT_TIMESTAMP()
    WHERE job_run_id = '{JOB_RUN_ID}'
    """
)

display(
    spark.sql(
        f"""
        SELECT *
        FROM semiconplus_portfolio.operations.workflow_run_log
        WHERE job_run_id = '{JOB_RUN_ID}'
        """
    )
)

print("Workflow failure recorded.")